In [1]:
# ============================================
# MODULE 5: EXPLAINABLE AI - FAKE NEWS PREDICTOR
# ============================================

import pickle
import pandas as pd
import re
import nltk
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords

# Download stopwords if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')

stop_words = set(stopwords.words('english'))

In [3]:
# ============================================
# STEP 1: LOAD TRAINED MODEL FROM MODULE 1
# ============================================

print("="*60)
print("LOADING TRAINED MODEL")
print("="*60)

# Load the model trained in Module 1
try:
    model = pickle.load(
        open("../module 1/fake_news_model.pkl", "rb")
    )
    print("✓ Model loaded successfully")
except:
    # Alternative path if model is in same directory
    try:
        model = pickle.load(
            open("fake_news_model.pkl", "rb")
        )
        print("✓ Model loaded successfully")
    except:
        print("✗ Could not find model file. Please ensure model is saved.")
        print("  Expected path: ../module 1/fake_news_model.pkl")

# Load the vectorizer
try:
    vectorizer = pickle.load(
        open("../module 1/tfidf_vectorizer.pkl", "rb")
    )
    print("✓ Vectorizer loaded successfully")
except:
    try:
        vectorizer = pickle.load(
            open("tfidf_vectorizer.pkl", "rb")
        )
        print("✓ Vectorizer loaded successfully")
    except:
        print("✗ Could not find vectorizer file.")
        print("  Expected path: ../module 1/tfidf_vectorizer.pkl")

LOADING TRAINED MODEL
✓ Model loaded successfully
✓ Vectorizer loaded successfully


In [4]:
# ============================================
# STEP 2: CLEANING FUNCTION (SAME AS MODULE 1)
# ============================================

def clean_text(text):
    """
    Clean text for prediction (same as Module 1)
    """
    if pd.isna(text) or str(text).strip() == '':
        return ""
    
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [5]:
# ============================================
# STEP 3: EXPLAIN PREDICTION FUNCTION
# ============================================

def explain_prediction(news):
    """
    Predict and explain why the news is Fake or Real
    
    Parameters:
    -----------
    news : str
        The news article text to analyze
        
    Returns:
    --------
    dict : Prediction results with explanation
    """
    
    # Clean the text
    cleaned_news = clean_text(news)
    
    # Convert text into TF-IDF
    news_vector = vectorizer.transform([cleaned_news])
    
    # Prediction
    prediction = model.predict(news_vector)[0]
    
    # Confidence Score
    confidence = model.predict_proba(news_vector).max()
    
    # Get feature names
    feature_names = vectorizer.get_feature_names_out()
    
    # Get TF-IDF values
    vector_data = news_vector.toarray()[0]
    
    # Store important words
    important_words = []
    
    for index, value in enumerate(vector_data):
        if value > 0:
            important_words.append(
                (
                    feature_names[index],
                    value
                )
            )
    
    # Sort words by importance
    important_words = sorted(
        important_words,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Top 5 important words
    top_words = important_words[:5]
    
    # Get all important words with scores
    all_important_words = [(word, round(score, 4)) for word, score in important_words[:10]]
    
    # Determine label
    label = "Fake" if prediction == 1 else "Real"
    
    # Create reason
    reason_words = [word for word, score in top_words]
    reason = f"These words contributed to {label} News prediction: {', '.join(reason_words)}"
    
    # Output results
    print("="*60)
    print("FAKE NEWS PREDICTOR - EXPLANATION")
    print("="*60)
    
    print(f"\n📰 News Article:")
    print(f"   {news[:200]}..." if len(news) > 200 else f"   {news}")
    
    print(f"\n📊 Prediction: {label}")
    print(f"   Confidence: {round(confidence * 100, 2)}%")
    
    print(f"\n🔑 Important Words:")
    for word, score in top_words:
        print(f"   • {word}: {round(score, 4)}")
    
    print(f"\n💡 Reason:")
    print(f"   {reason}")
    
    print("\n" + "="*60)
    
    # Return results as dictionary
    return {
        'news': news,
        'cleaned_news': cleaned_news,
        'prediction': prediction,
        'label': label,
        'confidence': round(confidence * 100, 2),
        'top_words': top_words,
        'all_important_words': all_important_words,
        'reason': reason
    }

In [6]:
# ============================================
# STEP 4: TEST WITH SAMPLE NEWS
# ============================================

print("\n" + "="*60)
print("TESTING PREDICTOR WITH SAMPLE NEWS")
print("="*60)

sample_news = [
    "Breaking Secret Government Plan Revealed",
    "Pakistan cricket team won the match against India.",
    "Government announces new economic reforms for Pakistan.",
    "Apple launched a new AI-powered iPhone today.",
    "BREAKING SECRET SHOCKING NEWS REVEALED!!!"
]

for news in sample_news:
    result = explain_prediction(news)
    print("\n" + "-"*60 + "\n")


TESTING PREDICTOR WITH SAMPLE NEWS
FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   Breaking Secret Government Plan Revealed

📊 Prediction: Real
   Confidence: 75.6%

🔑 Important Words:
   • breaking: 0.5329
   • revealed: 0.4989
   • secret: 0.4899
   • plan: 0.3771
   • government: 0.2914

💡 Reason:
   These words contributed to Real News prediction: breaking, revealed, secret, plan, government


------------------------------------------------------------

FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   Pakistan cricket team won the match against India.

📊 Prediction: Real
   Confidence: 99.0%

🔑 Important Words:
   • cricket: 0.5342
   • match: 0.4604
   • pakistan: 0.4568
   • india: 0.4252
   • team: 0.3365

💡 Reason:
   These words contributed to Real News prediction: cricket, match, pakistan, india, team


------------------------------------------------------------

FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   Government announces new economic reforms for 

In [8]:
# ============================================
# STEP 5: BATCH PREDICTION WITH EXPLANATIONS
# ============================================

def batch_predict(news_list):
    """
    Predict and explain multiple news articles
    
    Parameters:
    -----------
    news_list : list
        List of news articles
        
    Returns:
    --------
    list : Results for each article
    """
    results = []
    
    print("\n" + "="*60)
    print("BATCH PREDICTION")
    print("="*60)
    
    for i, news in enumerate(news_list, 1):
        print(f"\n📌 Article {i}:")
        result = explain_prediction(news)
        results.append(result)
        print("-"*40)
    
    # Summary
    fake_count = sum(1 for r in results if r['prediction'] == 1)
    real_count = sum(1 for r in results if r['prediction'] == 0)
    
    print("\n" + "="*60)
    print("BATCH SUMMARY")
    print("="*60)
    print(f"Total Articles: {len(results)}")
    print(f"Fake Predictions: {fake_count}")
    print(f"Real Predictions: {real_count}")
    print("="*60)
    
    return results

# Test batch prediction
news_batch = [
    "Breaking: Scientists discover cure for cancer!",
    "The Federal Reserve announced interest rate cuts today.",
    "Aliens landed in Lahore, government hiding truth!",
    "Pakistan's economy shows growth despite global challenges.",
    "SHOCKING: World leaders agree to end all wars!"
]

# Uncomment to run batch prediction
# batch_results = batch_predict(news_batch)

In [9]:
# ============================================
# STEP 6: PREDICT WITH SOURCE VERIFICATION
# ============================================

def predict_with_source_verification(news_text, source_url=None):
    """
    Enhanced prediction that combines fake news detection with source verification
    
    Parameters:
    -----------
    news_text : str
        The news article text
    source_url : str, optional
        The URL of the news source
        
    Returns:
    --------
    dict : Combined prediction results
    """
    
    # Get fake news prediction
    result = explain_prediction(news_text)
    
    # Add source verification if URL is provided
    if source_url:
        print("\n" + "="*60)
        print("SOURCE VERIFICATION")
        print("="*60)
        print(f"Source URL: {source_url}")
        
        # Import source verification from Module 4
        try:
            from urllib.parse import urlparse
            
            # Simple source verification
            domain = urlparse(source_url).netloc
            domain = domain.replace("www.", "")
            
            # Check if domain is known
            known_sources = ['reuters.com', 'bbc.com', 'dawn.com', 'geo.tv', 
                            'arynews.tv', 'cnn.com', 'aljazeera.com', 'nytimes.com']
            
            if domain in known_sources:
                print(f"✓ Domain '{domain}' is a known source")
                source_credibility = "High"
            else:
                print(f"⚠️ Domain '{domain}' is not in trusted sources")
                source_credibility = "Unknown"
            
            # Check HTTPS
            https_status = "Secure" if source_url.startswith("https") else "Not Secure"
            print(f"HTTPS: {https_status}")
            
            # Overall risk
            if result['label'] == 'Fake' and source_credibility == 'Unknown':
                risk = "HIGH - Suspicious content from unknown source"
            elif result['label'] == 'Fake':
                risk = "MEDIUM - Fake content from known source"
            else:
                risk = "LOW - Content appears legitimate"
            
            print(f"Risk Level: {risk}")
            
            result['source_url'] = source_url
            result['source_domain'] = domain
            result['source_credibility'] = source_credibility
            result['https_status'] = https_status
            result['risk'] = risk
            
        except Exception as e:
            print(f"Could not verify source: {e}")
    
    return result

In [10]:
# ============================================
# STEP 7: TEST WITH SOURCE VERIFICATION
# ============================================

print("\n" + "="*60)
print("TESTING WITH SOURCE VERIFICATION")
print("="*60)

# Test with source verification
result = predict_with_source_verification(
    "Breaking: Scientists discover miracle cure for all diseases!",
    "https://viral-news-xyz.com"
)

print("\n" + "="*60)
print("COMPLETE ANALYSIS")
print("="*60)
print(f"Prediction: {result['label']}")
print(f"Confidence: {result['confidence']}%")
print(f"Risk Level: {result.get('risk', 'Unknown')}")


TESTING WITH SOURCE VERIFICATION
FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   Breaking: Scientists discover miracle cure for all diseases!

📊 Prediction: Fake
   Confidence: 85.0%

🔑 Important Words:
   • diseases: 0.5726
   • discover: 0.5574
   • breaking: 0.4416
   • scientists: 0.4079

💡 Reason:
   These words contributed to Fake News prediction: diseases, discover, breaking, scientists


SOURCE VERIFICATION
Source URL: https://viral-news-xyz.com
⚠️ Domain 'viral-news-xyz.com' is not in trusted sources
HTTPS: Secure
Risk Level: HIGH - Suspicious content from unknown source

COMPLETE ANALYSIS
Prediction: Fake
Confidence: 85.0%
Risk Level: HIGH - Suspicious content from unknown source


In [12]:
# ============================================
# STEP 8: SAVE PREDICTIONS WITH EXPLANATIONS
# ============================================

def save_predictions_with_explanations(predictions, filename="predictions_with_explanations.csv"):
    """
    Save prediction results to CSV
    
    Parameters:
    -----------
    predictions : list
        List of prediction results
    filename : str
        Output CSV filename
    """
    
    # Convert to DataFrame
    df_results = pd.DataFrame(predictions)
    
    # Save to CSV
    df_results.to_csv(filename, index=False)
    print(f"✓ Predictions saved to: {filename}")
    print(f"  Total predictions: {len(df_results)}")
    
    return df_results

# Example: Save results
# save_predictions_with_explanations(batch_results)

In [14]:
# ============================================
# STEP 9: LOAD AND USE INDIVIDUAL DATASETS
# ============================================

print("\n" + "="*60)
print("LOADING DATASETS FOR BATCH PREDICTION")
print("="*60)

# Load the datasets directly
datasets = []

try:
    fake = pd.read_csv("../datasets/fake.csv")
    fake['label'] = 0
    datasets.append(fake)
    print(f"✓ Loaded fake.csv: {fake.shape[0]:,} articles")
except Exception as e:
    print(f"✗ Could not load fake.csv: {e}")

try:
    true = pd.read_csv("../datasets/True.csv")
    true['label'] = 1
    datasets.append(true)
    print(f"✓ Loaded True.csv: {true.shape[0]:,} articles")
except Exception as e:
    print(f"✗ Could not load True.csv: {e}")

try:
    ag_train = pd.read_csv("../datasets/AG_Train.csv")
    print(f"✓ Loaded AG_Train.csv: {ag_train.shape[0]:,} articles")
except Exception as e:
    print(f"✗ Could not load AG_Train.csv: {e}")

try:
    ag_test = pd.read_csv("../datasets/AG_Test.csv")
    print(f"✓ Loaded AG_Test.csv: {ag_test.shape[0]:,} articles")
except Exception as e:
    print(f"✗ Could not load AG_Test.csv: {e}")

# Combine all datasets
if datasets:
    df = pd.concat(datasets, ignore_index=True)
    print(f"\n✅ Total combined: {len(df):,} articles")
    
    # Create text column if it doesn't exist
    if 'text' not in df.columns:
        if 'title' in df.columns and 'description' in df.columns:
            df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
        elif 'title' in df.columns:
            df['text'] = df['title'].fillna('')
        else:
            # Try to find any text column
            for col in ['content', 'article', 'body', 'description']:
                if col in df.columns:
                    df['text'] = df[col]
                    break
            else:
                df['text'] = df[df.columns[0]]

    # Sample 20 articles for prediction
    sample_df = df.sample(n=20, random_state=42)
    
    print("\n" + "="*60)
    print("PREDICTING ON SAMPLE ARTICLES")
    print("="*60)
    
    predictions_list = []
    
    for idx, row in sample_df.iterrows():
        news_text = row.get('text', row.get('clean_text', ''))
        if news_text and len(str(news_text).strip()) > 10:
            print(f"\n📌 Article {idx}")
            result = explain_prediction(news_text)
            predictions_list.append(result)
            print("-"*60)
    
    # Save predictions
    if predictions_list:
        df_results = pd.DataFrame(predictions_list)
        df_results.to_csv("predictions_with_explanations.csv", index=False)
        print(f"\n✅ Predictions saved to: predictions_with_explanations.csv")
        print(f"   Total predictions: {len(df_results)}")
        
        # Summary
        fake_count = sum(1 for r in predictions_list if r['prediction'] == 1)
        real_count = sum(1 for r in predictions_list if r['prediction'] == 0)
        print(f"\n📊 Summary:")
        print(f"   Real Predictions: {real_count}")
        print(f"   Fake Predictions: {fake_count}")
else:
    print("❌ No datasets could be loaded!")


LOADING DATASETS FOR BATCH PREDICTION
✓ Loaded fake.csv: 23,481 articles
✓ Loaded True.csv: 21,417 articles
✓ Loaded AG_Train.csv: 120,000 articles
✓ Loaded AG_Test.csv: 7,600 articles

✅ Total combined: 44,898 articles

PREDICTING ON SAMPLE ARTICLES

📌 Article 22216
FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as Ferris Bueller s Day Off) made some provocative s...

📊 Prediction: Real
   Confidence: 96.0%

🔑 Important Words:
   • stein: 0.2716
   • century wire: 0.1844
   • st century: 0.1768
   • halt: 0.1738
   • note: 0.172

💡 Reason:
   These words contributed to Real News prediction: stein, century wire, st century, halt, note

------------------------------------------------------------

📌 Article 27917
FAKE NEWS PREDICTOR - EXPLANATION

📰 News Article:
   WASHINGTON (Reuters) - U.S. President Donald Trump removed his chief

In [15]:
# ============================================
# STEP 10: INTERACTIVE PREDICTION FUNCTION
# ============================================

def interactive_predict():
    """
    Interactive mode for predicting news articles
    """
    print("\n" + "="*60)
    print("INTERACTIVE FAKE NEWS PREDICTOR")
    print("="*60)
    print("\nEnter a news article to predict (type 'quit' to exit)")
    
    while True:
        print("\n" + "-"*60)
        news = input("Enter news text: ")
        
        if news.lower() == 'quit':
            print("\nGoodbye!")
            break
        
        if not news.strip():
            print("Please enter some text.")
            continue
        
        # Get prediction
        result = explain_prediction(news)
        
        print("\n" + "="*60)
        print("Would you like to add source verification? (y/n)")
        add_source = input().lower()
        
        if add_source == 'y':
            url = input("Enter source URL: ")
            result = predict_with_source_verification(news, url)
        
        print("\n" + "="*60)
        print("To continue, enter another article or type 'quit'")

# Uncomment to run interactive mode
# interactive_predict()

In [17]:
# ============================================
# STEP 11: FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("MODULE 5 COMPLETE - EXPLAINABLE AI")
print("="*60)

print(f"""
Module 5: Explainable AI - Fake News Predictor

Functions Created:
✓ explain_prediction() - Predict and explain individual news
✓ batch_predict() - Predict multiple articles at once
✓ predict_with_source_verification() - Combine with source verification
✓ interactive_predict() - Interactive mode for predictions
✓ save_predictions_with_explanations() - Save results to CSV

Key Features:
• Shows important words in prediction
• Provides confidence score
• Explains why news is Fake or Real
• Batch prediction support
• Source verification integration
• Interactive mode

How It Works:
1. News text is cleaned (same as Module 1)
2. Converted to TF-IDF vectors
3. Model predicts Fake or Real
4. Important words are extracted
5. Explanation is provided

Usage Examples:
--------------
# Predict a single article
result = explain_prediction("Breaking news about aliens!")

# Batch predict multiple articles
results = batch_predict(["news1", "news2", "news3"])

# Predict with source verification
result = predict_with_source_verification(news, "https://bbc.com")

Model Used: {type(model).__name__}
Accuracy: 95%+ (from Module 1 training)
""")

print("="*60)
print("MODULE 5 COMPLETE!")
print("="*60)


MODULE 5 COMPLETE - EXPLAINABLE AI

Module 5: Explainable AI - Fake News Predictor

Functions Created:
✓ explain_prediction() - Predict and explain individual news
✓ batch_predict() - Predict multiple articles at once
✓ predict_with_source_verification() - Combine with source verification
✓ interactive_predict() - Interactive mode for predictions
✓ save_predictions_with_explanations() - Save results to CSV

Key Features:
• Shows important words in prediction
• Provides confidence score
• Explains why news is Fake or Real
• Batch prediction support
• Source verification integration
• Interactive mode

How It Works:
1. News text is cleaned (same as Module 1)
2. Converted to TF-IDF vectors
3. Model predicts Fake or Real
4. Important words are extracted
5. Explanation is provided

Usage Examples:
--------------
# Predict a single article
result = explain_prediction("Breaking news about aliens!")

# Batch predict multiple articles
results = batch_predict(["news1", "news2", "news3"])

# Pre